In [2]:
import pandas as pd
df = pd.read_csv("scales.csv")
print(df.columns)

Index(['TrumpGen', 'TrumpAbort', 'SCGen', 'SCAbort', 'BidenGen', 'BidenAbort',
       'religion', 'bornagain', 'CRS1', 'CRS2', 'CRS3', 'CRS4', 'CRS5',
       'AbortionWrong', 'AbortionMurder', 'AbortionRape', 'AbortionHealth',
       'AbortionDefect', 'AbortionWant', 'AbortionAll', 'AbortionWait',
       'AbortionUltrasound', 'PolAffiliate', 'IdeologySocial',
       'IdeologyEconomy', 'IdeologySecurity', 'vote2020', 'Acheck3'],
      dtype='object')


In [6]:
"""
RQ1 Analysis: How does AI-mediated search transform attitude-related divergence
in comparison to traditional search?

Methodology follows the instructions document + Habib et al. (2025) paper.

DATA STATUS:
- Google data has significant Selenium scraping failures (participants 147-227)
- Chicken: 41/227 valid Google results
- States: 14/227 valid Google results  
- Breast Cancer: 2/227 valid Google results (UNUSABLE)
- Fetus: NO Google CSV available
- ChatGPT: All 227 participants have responses for all tasks
- Participant grouping: APPROXIMATE (awaiting author mapping)

ANALYSIS DIMENSIONS:
A) Text similarity divergence (TF-IDF vocabulary, following paper §3.3 and §4.1)
B) Text polarity divergence (keyword-based, approximating paper's FastText method §3.4)
C) Domain divergence (where available)
D) Distribution-based metrics: JS divergence, Δmeans with bootstrap CI
E) Similarity-based metrics: Δsim, cohesion, separation with KS tests
"""

import json
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp
import re
from collections import Counter

warnings.filterwarnings('ignore')
np.random.seed(42)

# =============================================================================
# DATA LOADING AND CLEANING
# =============================================================================

def load_and_clean_participants(filepath):
    """Load participant data and compute attitude groupings."""
    df = pd.read_csv(filepath, skiprows=[1])
    
    abort_cols = ['AbortionWrong', 'AbortionMurder', 'AbortionRape', 'AbortionHealth', 
                  'AbortionDefect', 'AbortionWant', 'AbortionAll', 'AbortionWait', 'AbortionUltrasound']
    
    for col in abort_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Reverse code starred items so higher = more pro-choice (paper §3.1)
    reverse_cols = ['AbortionWrong', 'AbortionMurder', 'AbortionWait', 'AbortionUltrasound']
    for col in reverse_cols:
        df[col + '_rev'] = 8 - df[col]
    
    agg_cols = [c + '_rev' if c in reverse_cols else c for c in abort_cols]
    df['abortion_agg'] = df[agg_cols].mean(axis=1)
    
    # Grouping: < 3 = pro-life, > 5 = pro-choice (paper methodology)
    df['attitude_group'] = 'neutral'
    df.loc[df['abortion_agg'] < 3, 'attitude_group'] = 'pro_life'
    df.loc[df['abortion_agg'] > 5, 'attitude_group'] = 'pro_choice'
    
    df['query_index'] = range(1, len(df) + 1)
    
    # Flag valid queries (> 3 chars, not single letter junk)
    for col in ['NPC', 'ALowC', 'AHighC', 'MisC']:
        df[col + '_valid'] = df[col].apply(lambda x: isinstance(x, str) and len(str(x).strip()) > 3)
    
    return df


def load_google_clean(filepath):
    """Load and clean Google results - remove errors and NaN entries."""
    df = pd.read_csv(filepath)
    
    # Remove ERROR rows
    error_mask = df['title'].str.contains('ERROR', na=False)
    df_clean = df[~error_mask].copy()
    
    # Remove rows where title is NaN (no actual result)
    df_clean = df_clean[df_clean['title'].notna()].copy()
    
    return df_clean


def load_chatgpt(filepath):
    """Load ChatGPT results."""
    with open(filepath) as f:
        data = json.load(f)
    return data


# =============================================================================
# TEXT EXTRACTION
# =============================================================================

def get_google_text(google_df, query_index):
    """Concatenate titles + snippets for a participant's Google results."""
    rows = google_df[google_df['query_index'] == query_index]
    parts = []
    for _, row in rows.iterrows():
        t = str(row.get('title', ''))
        s = str(row.get('snippet', ''))
        if t != 'nan': parts.append(t)
        if s != 'nan': parts.append(s)
    return ' '.join(parts)


def get_google_domains(google_df, query_index):
    """Get domains from Google results."""
    rows = google_df[google_df['query_index'] == query_index]
    return [str(d) for d in rows['registrable_domain'].dropna().tolist() if str(d) != 'nan']


def get_chatgpt_text(chatgpt_data, query_index):
    """Get ChatGPT response text."""
    return chatgpt_data[query_index - 1].get('answer_text_raw', '')


def get_chatgpt_domains(chatgpt_data, query_index):
    """Get cited domains from ChatGPT response."""
    entry = chatgpt_data[query_index - 1]
    return [c.get('registrable_domain', '') for c in entry.get('citations', []) if c.get('registrable_domain')]


# =============================================================================
# POLARITY SCORING (approximating paper's FastText method)
# =============================================================================

# Extended keyword lists based on Table 8 of the paper and common vocabulary
PRO_LIFE_WORDS = {
    'unborn', 'murder', 'conception', 'baby', 'babies', 'innocent', 'sanctity', 
    'heartbeat', 'kill', 'killing', 'womb', 'protect', 'morally', 'wrong', 
    'sin', 'immoral', 'ban', 'bans', 'banned', 'outlawed', 'outlaw',
    'criminalize', 'prohibit', 'restrict', 'condemned', 'pro-life', 'prolife',
    'birth', 'born', 'life', 'illegal', 'crime', 'criminal', 'penalty',
    'personhood', 'child', 'children', 'survivor', 'viable', 'viability',
    'gestational', 'trimester', 'heartbeat', 'fertilization'
}

PRO_CHOICE_WORDS = {
    'rights', 'right', 'choice', 'choose', 'fetus', 'fetal', 'freedom', 
    'autonomy', 'bodily', 'reproductive', 'access', 'healthcare', 'health',
    'legal', 'legality', 'legalize', 'women', 'woman', 'care', 'clinic', 
    'clinics', 'provider', 'safe', 'constitutional', 'privacy', 'roe', 'wade',
    'planned', 'parenthood', 'pro-choice', 'prochoice', 'reproductive',
    'maternal', 'pregnancy', 'pregnant', 'trimester', 'gestation',
    'procedure', 'medical', 'medication', 'pill', 'mifepristone'
}


def compute_text_polarity(text):
    """
    Compute abortion-specific polarity for a single text.
    Positive = pro-life leaning, Negative = pro-choice leaning.
    """
    if not text or not text.strip():
        return 0.0
    words = set(re.findall(r'\b\w+\b', text.lower()))
    pl = len(words & PRO_LIFE_WORDS)
    pc = len(words & PRO_CHOICE_WORDS)
    total = pl + pc
    if total == 0:
        return 0.0
    return (pl - pc) / total


# =============================================================================
# DIVERGENCE METRICS
# =============================================================================

def compute_similarity_divergence(group1_texts, group2_texts, n_bootstrap=2000):
    """
    Similarity-based divergence following paper's Equation (1) and methodology.
    Uses TF-IDF vectors + cosine similarity.
    
    Returns: pct_diff, delta_sim, within/across means, KS test, bootstrap CI,
             cohesion, separation
    """
    # Filter empty
    g1 = [t for t in group1_texts if t and t.strip() and len(t.strip()) > 10]
    g2 = [t for t in group2_texts if t and t.strip() and len(t.strip()) > 10]
    
    if len(g1) < 3 or len(g2) < 3:
        return {'status': 'INSUFFICIENT_DATA', 'n_g1': len(g1), 'n_g2': len(g2)}
    
    all_texts = g1 + g2
    n1, n2 = len(g1), len(g2)
    
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', min_df=1)
    try:
        vectors = vectorizer.fit_transform(all_texts).toarray()
    except ValueError:
        return {'status': 'VECTORIZATION_FAILED', 'n_g1': n1, 'n_g2': n2}
    
    sim_matrix = cosine_similarity(vectors)
    
    # Within-group similarities
    within_g1 = [sim_matrix[i][j] for i in range(n1) for j in range(i+1, n1)]
    within_g2 = [sim_matrix[i][j] for i in range(n1, n1+n2) for j in range(i+1, n1+n2)]
    within_all = within_g1 + within_g2
    
    # Across-group similarities
    across = [sim_matrix[i][j] for i in range(n1) for j in range(n1, n1+n2)]
    
    mean_within = np.mean(within_all)
    mean_across = np.mean(across)
    delta_sim = mean_within - mean_across
    pct_diff = (delta_sim / mean_within * 100) if mean_within != 0 else np.nan
    
    # KS test
    ks_stat, ks_pval = ks_2samp(within_all, across)
    
    # Bootstrap CI for delta_sim
    within_arr = np.array(within_all)
    across_arr = np.array(across)
    boot_deltas = []
    for _ in range(n_bootstrap):
        bw = np.random.choice(within_arr, len(within_arr), replace=True)
        ba = np.random.choice(across_arr, len(across_arr), replace=True)
        boot_deltas.append(np.mean(bw) - np.mean(ba))
    
    ci_low, ci_high = np.percentile(boot_deltas, [2.5, 97.5])
    
    return {
        'status': 'OK',
        'n_g1': n1, 'n_g2': n2,
        'mean_within': mean_within,
        'mean_across': mean_across,
        'delta_sim': delta_sim,
        'pct_diff': pct_diff,
        'ks_stat': ks_stat,
        'ks_pval': ks_pval,
        'ks_significant': ks_pval < 0.05,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'bootstrap_significant': ci_low > 0,
        'cohesion_g1': np.mean(within_g1) if within_g1 else np.nan,
        'cohesion_g2': np.mean(within_g2) if within_g2 else np.nan,
        'separation': mean_across
    }


def compute_distribution_divergence(values1, values2, n_bootstrap=2000):
    """
    Distribution-based divergence: JS divergence + Δmeans with bootstrap CI.
    """
    v1 = np.array([v for v in values1 if not np.isnan(v)])
    v2 = np.array([v for v in values2 if not np.isnan(v)])
    
    if len(v1) < 3 or len(v2) < 3:
        return {'status': 'INSUFFICIENT_DATA'}
    
    # JS Divergence
    all_vals = np.concatenate([v1, v2])
    n_bins = min(30, max(5, len(all_vals) // 5))
    bins = np.linspace(all_vals.min() - 0.01, all_vals.max() + 0.01, n_bins)
    
    h1, _ = np.histogram(v1, bins=bins, density=True)
    h2, _ = np.histogram(v2, bins=bins, density=True)
    h1 = h1 + 1e-10; h2 = h2 + 1e-10
    h1 /= h1.sum(); h2 /= h2.sum()
    
    js_div = jensenshannon(h1, h2) ** 2
    
    # Δmeans
    delta_means = np.mean(v1) - np.mean(v2)
    
    # Bootstrap CI
    boot_diffs = []
    for _ in range(n_bootstrap):
        b1 = np.random.choice(v1, len(v1), replace=True)
        b2 = np.random.choice(v2, len(v2), replace=True)
        boot_diffs.append(np.mean(b1) - np.mean(b2))
    
    ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
    
    return {
        'status': 'OK',
        'mean_g1': np.mean(v1), 'mean_g2': np.mean(v2),
        'std_g1': np.std(v1), 'std_g2': np.std(v2),
        'n_g1': len(v1), 'n_g2': len(v2),
        'js_div': js_div,
        'delta_means': delta_means,
        'ci_low': ci_low, 'ci_high': ci_high,
        'significant': not (ci_low <= 0 <= ci_high)
    }


def compute_domain_divergence(domains_g1, domains_g2):
    """
    Domain-based divergence using frequency vectors + cosine similarity.
    """
    # Filter empty
    dg1 = [d for d in domains_g1 if len(d) > 0]
    dg2 = [d for d in domains_g2 if len(d) > 0]
    
    if len(dg1) < 3 or len(dg2) < 3:
        return {'status': 'INSUFFICIENT_DATA', 'n_g1': len(dg1), 'n_g2': len(dg2)}
    
    # Build domain vocabulary
    all_domains = sorted(set(d for dl in dg1 + dg2 for d in dl))
    if len(all_domains) < 2:
        return {'status': 'INSUFFICIENT_DOMAINS'}
    
    def freq_vec(domain_list):
        c = Counter(domain_list)
        return [c.get(d, 0) for d in all_domains]
    
    vecs1 = np.array([freq_vec(dl) for dl in dg1], dtype=float)
    vecs2 = np.array([freq_vec(dl) for dl in dg2], dtype=float)
    
    all_vecs = np.vstack([vecs1, vecs2])
    n1, n2 = len(vecs1), len(vecs2)
    
    sim_mat = cosine_similarity(all_vecs)
    
    within = ([sim_mat[i][j] for i in range(n1) for j in range(i+1, n1)] +
              [sim_mat[i][j] for i in range(n1, n1+n2) for j in range(i+1, n1+n2)])
    across = [sim_mat[i][j] for i in range(n1) for j in range(n1, n1+n2)]
    
    mean_w = np.mean(within) if within else 0
    mean_a = np.mean(across) if across else 0
    pct_diff = ((mean_w - mean_a) / mean_w * 100) if mean_w != 0 else np.nan
    
    ks_stat, ks_pval = ks_2samp(within, across) if within and across else (np.nan, np.nan)
    
    return {
        'status': 'OK', 'n_g1': n1, 'n_g2': n2,
        'n_unique_domains': len(all_domains),
        'mean_within': mean_w, 'mean_across': mean_a,
        'pct_diff': pct_diff,
        'ks_stat': ks_stat, 'ks_pval': ks_pval,
        'ks_significant': ks_pval < 0.05 if not np.isnan(ks_pval) else False
    }


# =============================================================================
# ANALYSIS RUNNER
# =============================================================================

def analyze_task(task_name, participants, chatgpt_data, google_df, query_col):
    """Full divergence analysis for one task, both modalities."""
    
    print(f"\n{'='*80}")
    print(f" TASK: {task_name}")
    print(f"{'='*80}")
    
    pl = participants[participants['attitude_group'] == 'pro_life']
    pc = participants[participants['attitude_group'] == 'pro_choice']
    
    # Filter valid queries
    pl_valid = pl[pl[query_col + '_valid']]
    pc_valid = pc[pc[query_col + '_valid']]
    
    # Participants with valid Google results
    google_participants = set(google_df['query_index'].unique())
    pl_google_idx = [idx for idx in pl_valid['query_index'] if idx in google_participants]
    pc_google_idx = [idx for idx in pc_valid['query_index'] if idx in google_participants]
    
    print(f"\n  Participants:")
    print(f"    Pro-life:  {len(pl)} total → {len(pl_valid)} valid queries → {len(pl_google_idx)} with Google data")
    print(f"    Pro-choice: {len(pc)} total → {len(pc_valid)} valid queries → {len(pc_google_idx)} with Google data")
    
    results = {'task': task_name}
    
    # ── ChatGPT texts ──
    c_pl_texts = [get_chatgpt_text(chatgpt_data, idx) for idx in pl_valid['query_index']]
    c_pc_texts = [get_chatgpt_text(chatgpt_data, idx) for idx in pc_valid['query_index']]
    
    # ── Google texts ──
    g_pl_texts = [get_google_text(google_df, idx) for idx in pl_google_idx]
    g_pc_texts = [get_google_text(google_df, idx) for idx in pc_google_idx]
    
    # ── ChatGPT polarities ──
    c_pl_pol = [compute_text_polarity(t) for t in c_pl_texts]
    c_pc_pol = [compute_text_polarity(t) for t in c_pc_texts]
    
    # ── Google polarities ──
    g_pl_pol = [compute_text_polarity(t) for t in g_pl_texts]
    g_pc_pol = [compute_text_polarity(t) for t in g_pc_texts]
    
    # ── ChatGPT domains ──
    c_pl_doms = [get_chatgpt_domains(chatgpt_data, idx) for idx in pl_valid['query_index']]
    c_pc_doms = [get_chatgpt_domains(chatgpt_data, idx) for idx in pc_valid['query_index']]
    
    # ── Google domains ──
    g_pl_doms = [get_google_domains(google_df, idx) for idx in pl_google_idx]
    g_pc_doms = [get_google_domains(google_df, idx) for idx in pc_google_idx]
    
    # =====================================================================
    # A) TEXT SIMILARITY DIVERGENCE
    # =====================================================================
    print(f"\n  ── A. Text Similarity Divergence (TF-IDF + Cosine) ──")
    
    # ChatGPT
    c_sim = compute_similarity_divergence(c_pl_texts, c_pc_texts)
    results['chatgpt_text_sim'] = c_sim
    if c_sim['status'] == 'OK':
        sig = '***' if c_sim['ks_pval'] < 0.001 else '**' if c_sim['ks_pval'] < 0.01 else '*' if c_sim['ks_pval'] < 0.05 else ''
        print(f"    ChatGPT (n={c_sim['n_g1']}+{c_sim['n_g2']}):")
        print(f"      Within: {c_sim['mean_within']:.4f}  Across: {c_sim['mean_across']:.4f}")
        print(f"      Pct diff: {c_sim['pct_diff']:.2f}%  KS p={c_sim['ks_pval']:.4f} {sig}")
        print(f"      Δsim: {c_sim['delta_sim']:.4f} CI[{c_sim['ci_low']:.4f}, {c_sim['ci_high']:.4f}]")
        print(f"      Cohesion G_l: {c_sim['cohesion_g1']:.4f}  G_c: {c_sim['cohesion_g2']:.4f}  Sep: {c_sim['separation']:.4f}")
    else:
        print(f"    ChatGPT: {c_sim['status']} (n={c_sim.get('n_g1', '?')}, {c_sim.get('n_g2', '?')})")
    
    # Google
    g_sim = compute_similarity_divergence(g_pl_texts, g_pc_texts)
    results['google_text_sim'] = g_sim
    if g_sim['status'] == 'OK':
        sig = '***' if g_sim['ks_pval'] < 0.001 else '**' if g_sim['ks_pval'] < 0.01 else '*' if g_sim['ks_pval'] < 0.05 else ''
        print(f"    Google  (n={g_sim['n_g1']}+{g_sim['n_g2']}):")
        print(f"      Within: {g_sim['mean_within']:.4f}  Across: {g_sim['mean_across']:.4f}")
        print(f"      Pct diff: {g_sim['pct_diff']:.2f}%  KS p={g_sim['ks_pval']:.4f} {sig}")
        print(f"      Δsim: {g_sim['delta_sim']:.4f} CI[{g_sim['ci_low']:.4f}, {g_sim['ci_high']:.4f}]")
        print(f"      Cohesion G_l: {g_sim['cohesion_g1']:.4f}  G_c: {g_sim['cohesion_g2']:.4f}  Sep: {g_sim['separation']:.4f}")
    else:
        print(f"    Google:  {g_sim['status']} (n={g_sim.get('n_g1', '?')}, {g_sim.get('n_g2', '?')})")
    
    # =====================================================================
    # B) TEXT POLARITY DIVERGENCE
    # =====================================================================
    print(f"\n  ── B. Text Polarity Divergence ──")
    
    # ChatGPT
    c_pol = compute_distribution_divergence(c_pl_pol, c_pc_pol)
    results['chatgpt_polarity'] = c_pol
    if c_pol['status'] == 'OK':
        sig = '*' if c_pol['significant'] else ''
        print(f"    ChatGPT (n={c_pol['n_g1']}+{c_pol['n_g2']}):")
        print(f"      G_l mean: {c_pol['mean_g1']:.4f} (σ={c_pol['std_g1']:.4f})")
        print(f"      G_c mean: {c_pol['mean_g2']:.4f} (σ={c_pol['std_g2']:.4f})")
        print(f"      JS div: {c_pol['js_div']:.4f}")
        print(f"      Δmeans: {c_pol['delta_means']:.4f} CI[{c_pol['ci_low']:.4f}, {c_pol['ci_high']:.4f}] {sig}")
    else:
        print(f"    ChatGPT: {c_pol['status']}")
    
    # Google
    g_pol = compute_distribution_divergence(g_pl_pol, g_pc_pol)
    results['google_polarity'] = g_pol
    if g_pol['status'] == 'OK':
        sig = '*' if g_pol['significant'] else ''
        print(f"    Google  (n={g_pol['n_g1']}+{g_pol['n_g2']}):")
        print(f"      G_l mean: {g_pol['mean_g1']:.4f} (σ={g_pol['std_g1']:.4f})")
        print(f"      G_c mean: {g_pol['mean_g2']:.4f} (σ={g_pol['std_g2']:.4f})")
        print(f"      JS div: {g_pol['js_div']:.4f}")
        print(f"      Δmeans: {g_pol['delta_means']:.4f} CI[{g_pol['ci_low']:.4f}, {g_pol['ci_high']:.4f}] {sig}")
    else:
        print(f"    Google:  {g_pol['status']}")
    
    # =====================================================================
    # C) DOMAIN DIVERGENCE
    # =====================================================================
    print(f"\n  ── C. Domain Divergence ──")
    
    # ChatGPT
    c_dom = compute_domain_divergence(c_pl_doms, c_pc_doms)
    results['chatgpt_domain'] = c_dom
    if c_dom['status'] == 'OK':
        sig = '*' if c_dom['ks_significant'] else ''
        print(f"    ChatGPT (n={c_dom['n_g1']}+{c_dom['n_g2']}, {c_dom['n_unique_domains']} unique domains):")
        print(f"      Within: {c_dom['mean_within']:.4f}  Across: {c_dom['mean_across']:.4f}")
        print(f"      Pct diff: {c_dom['pct_diff']:.2f}%  KS p={c_dom['ks_pval']:.4f} {sig}")
    else:
        print(f"    ChatGPT: {c_dom['status']} (n={c_dom.get('n_g1', '?')}, {c_dom.get('n_g2', '?')})")
    
    # Google
    g_dom = compute_domain_divergence(g_pl_doms, g_pc_doms)
    results['google_domain'] = g_dom
    if g_dom['status'] == 'OK':
        sig = '*' if g_dom['ks_significant'] else ''
        print(f"    Google  (n={g_dom['n_g1']}+{g_dom['n_g2']}, {g_dom['n_unique_domains']} unique domains):")
        print(f"      Within: {g_dom['mean_within']:.4f}  Across: {g_dom['mean_across']:.4f}")
        print(f"      Pct diff: {g_dom['pct_diff']:.2f}%  KS p={g_dom['ks_pval']:.4f} {sig}")
    else:
        print(f"    Google:  {g_dom['status']} (n={g_dom.get('n_g1', '?')}, {g_dom.get('n_g2', '?')})")
    
    # =====================================================================
    # COMPARISON
    # =====================================================================
    print(f"\n  ── COMPARISON ──")
    
    comparisons = []
    
    for metric_name, c_key, g_key, val_key in [
        ('Text Sim %diff', 'chatgpt_text_sim', 'google_text_sim', 'pct_diff'),
        ('Polarity JS Div', 'chatgpt_polarity', 'google_polarity', 'js_div'),
        ('Polarity Δmeans', 'chatgpt_polarity', 'google_polarity', 'delta_means'),
    ]:
        c_val = results.get(c_key, {}).get(val_key, np.nan) if results.get(c_key, {}).get('status') == 'OK' else np.nan
        g_val = results.get(g_key, {}).get(val_key, np.nan) if results.get(g_key, {}).get('status') == 'OK' else np.nan
        
        if not np.isnan(c_val) and not np.isnan(g_val):
            if abs(c_val) > abs(g_val):
                direction = 'AI > Search'
            elif abs(c_val) < abs(g_val):
                direction = 'AI < Search'
            else:
                direction = 'EQUAL'
            print(f"    {metric_name:<25} Google: {g_val:>10.4f}   ChatGPT: {c_val:>10.4f}   {direction}")
            comparisons.append((metric_name, g_val, c_val, direction))
        else:
            missing = []
            if np.isnan(g_val): missing.append('Google')
            if np.isnan(c_val): missing.append('ChatGPT')
            print(f"    {metric_name:<25} INCOMPLETE ({', '.join(missing)} data insufficient)")
    
    results['comparisons'] = comparisons
    return results


# =============================================================================
# CHATGPT-ONLY DEEP ANALYSIS (since Google data is sparse)
# =============================================================================

def chatgpt_deep_analysis(task_name, participants, chatgpt_data, query_col):
    """
    Deeper ChatGPT analysis examining response characteristics by group.
    This doesn't require Google data so we can analyze all participants.
    """
    print(f"\n  ── ChatGPT Deep Analysis ──")
    
    pl = participants[(participants['attitude_group'] == 'pro_life') & (participants[query_col + '_valid'])]
    pc = participants[(participants['attitude_group'] == 'pro_choice') & (participants[query_col + '_valid'])]
    neu = participants[(participants['attitude_group'] == 'neutral') & (participants[query_col + '_valid'])]
    
    # Response characteristics
    for group_name, group_df in [('Pro-life', pl), ('Pro-choice', pc), ('Neutral', neu)]:
        texts = [get_chatgpt_text(chatgpt_data, idx) for idx in group_df['query_index']]
        word_counts = [len(t.split()) for t in texts if t]
        polarities = [compute_text_polarity(t) for t in texts if t]
        
        citations = [chatgpt_data[idx-1].get('n_citations_found', 0) for idx in group_df['query_index']]
        
        print(f"    {group_name} (n={len(group_df)}):")
        print(f"      Avg word count: {np.mean(word_counts):.0f} ± {np.std(word_counts):.0f}")
        print(f"      Avg polarity:   {np.mean(polarities):.4f} ± {np.std(polarities):.4f}")
        print(f"      Avg citations:  {np.mean(citations):.1f}")
    
    # Query vocabulary comparison (replicating paper's RQ2 approach)
    pl_queries = [str(row[query_col]) for _, row in pl.iterrows()]
    pc_queries = [str(row[query_col]) for _, row in pc.iterrows()]
    
    q_sim = compute_similarity_divergence(pl_queries, pc_queries)
    if q_sim['status'] == 'OK':
        sig = '*' if q_sim['ks_significant'] else ''
        print(f"    Query vocabulary divergence:")
        print(f"      Pct diff: {q_sim['pct_diff']:.2f}%  KS p={q_sim['ks_pval']:.4f} {sig}")


# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    print("=" * 80)
    print(" RQ1: AI-mediated search vs Traditional search - Divergence Analysis")
    print("=" * 80)
    
    # Load participant data
    participants = load_and_clean_participants('final-dataset.csv')
    
    groups = participants['attitude_group'].value_counts()
    print(f"\nParticipant grouping (APPROXIMATE - awaiting author mapping):")
    print(f"  Pro-life:  {groups.get('pro_life', 0)}")
    print(f"  Pro-choice: {groups.get('pro_choice', 0)}")
    print(f"  Neutral:   {groups.get('neutral', 0)}")
    print(f"  Paper reports: Pro-life=72, Pro-choice=117, Neutral=38")
    
    # Task definitions
    tasks = [
        {
            'name': 'Chicken (Non-political control)',
            'chatgpt_file': 'chatgpt_chicken_results.json',
            'google_file': 'google_chicken_results.csv',
            'query_col': 'NPC'
        },
        {
            'name': 'States Outlawing Abortion (Political)',
            'chatgpt_file': 'chatgpt_states_results.json',
            'google_file': 'google_states_results.csv',
            'query_col': 'AHighC'
        },
        {
            'name': 'Fetus Survival (Political)',
            'chatgpt_file': 'chatgpt_fetus_results.json',
            'google_file': None,  # No Google CSV available
            'query_col': 'ALowC'
        },
        {
            'name': 'Breast Cancer Risk (Misinformation)',
            'chatgpt_file': 'chatgpt_breastcancer_results.json',
            'google_file': 'google_breastcancer_results.csv',
            'query_col': 'MisC'
        },
    ]
    
    all_results = {}
    
    for task in tasks:
        chatgpt_data = load_chatgpt(task['chatgpt_file'])
        
        if task['google_file']:
            google_df = load_google_clean(task['google_file'])
            n_google_valid = google_df['query_index'].nunique()
            print(f"\n  Google data: {n_google_valid} participants with valid results")
        else:
            google_df = pd.DataFrame(columns=['query_index', 'title', 'snippet', 'registrable_domain'])
            print(f"\n  Google data: NOT AVAILABLE")
        
        task_results = analyze_task(
            task['name'], participants, chatgpt_data, google_df, task['query_col']
        )
        
        # ChatGPT deep analysis (always available)
        chatgpt_deep_analysis(task['name'], participants, chatgpt_data, task['query_col'])
        
        all_results[task['name']] = task_results
    
    # =========================================================================
    # FINAL SUMMARY TABLE
    # =========================================================================
    print(f"\n\n{'='*80}")
    print(" SUMMARY TABLE")
    print(f"{'='*80}")
    
    header = f"{'Task':<40} {'Metric':<20} {'Google':<15} {'ChatGPT':<15} {'Comparison':<15}"
    print(header)
    print('-' * len(header))
    
    for task_name, res in all_results.items():
        first = True
        for metric_name, c_key, g_key, val_key, sig_key in [
            ('Text %diff', 'chatgpt_text_sim', 'google_text_sim', 'pct_diff', 'ks_significant'),
            ('Polarity JS', 'chatgpt_polarity', 'google_polarity', 'js_div', 'significant'),
            ('Polarity Δμ', 'chatgpt_polarity', 'google_polarity', 'delta_means', 'significant'),
        ]:
            c_res = res.get(c_key, {})
            g_res = res.get(g_key, {})
            
            c_val = c_res.get(val_key, np.nan) if c_res.get('status') == 'OK' else np.nan
            g_val = g_res.get(val_key, np.nan) if g_res.get('status') == 'OK' else np.nan
            
            c_sig = '*' if c_res.get(sig_key, False) else ''
            g_sig = '*' if g_res.get(sig_key, False) else ''
            
            c_str = f"{c_val:.4f}{c_sig}" if not np.isnan(c_val) else "N/A"
            g_str = f"{g_val:.4f}{g_sig}" if not np.isnan(g_val) else "N/A"
            
            if not np.isnan(c_val) and not np.isnan(g_val):
                comp = 'AI>Srch' if abs(c_val) > abs(g_val) else 'AI<Srch' if abs(c_val) < abs(g_val) else '='
            else:
                comp = '--'
            
            tname = task_name[:38] if first else ''
            first = False
            print(f"{tname:<40} {metric_name:<20} {g_str:<15} {c_str:<15} {comp}")
        print()
    
    print("\nNOTES:")
    print("  * = statistically significant (p < 0.05)")
    print("  Pct diff > 0: within-group more similar than across-group (divergence present)")
    print("  Pct diff < 0: across-group more similar than within-group (convergence)")
    print("  AI>Srch: ChatGPT shows more divergence than Google Search")
    print("  AI<Srch: ChatGPT shows less divergence than Google Search")
    print()
    print("DATA LIMITATIONS:")
    print(f"  - Participant grouping is APPROXIMATE (13 pro-life vs paper's 72)")
    print(f"  - Google scraping failed for participants 147-227 (Selenium errors)")
    print(f"  - Google data: Chicken={41}, States={14}, BreastCancer={2}, Fetus=missing")
    print(f"  - Google comparisons have very low power due to sparse data")
    print(f"  - ChatGPT data is complete (227 participants for all tasks)")

 RQ1: AI-mediated search vs Traditional search - Divergence Analysis

Participant grouping (APPROXIMATE - awaiting author mapping):
  Pro-life:  13
  Pro-choice: 96
  Neutral:   118
  Paper reports: Pro-life=72, Pro-choice=117, Neutral=38

  Google data: 41 participants with valid results

 TASK: Chicken (Non-political control)

  Participants:
    Pro-life:  13 total → 13 valid queries → 1 with Google data
    Pro-choice: 96 total → 96 valid queries → 21 with Google data

  ── A. Text Similarity Divergence (TF-IDF + Cosine) ──
    ChatGPT (n=13+96):
      Within: 0.2882  Across: 0.3079
      Pct diff: -6.83%  KS p=0.0000 ***
      Δsim: -0.0197 CI[-0.0280, -0.0112]
      Cohesion G_l: 0.3310  G_c: 0.2875  Sep: 0.3079
    Google:  INSUFFICIENT_DATA (n=1, 21)

  ── B. Text Polarity Divergence ──
    ChatGPT (n=13+96):
      G_l mean: -0.0769 (σ=0.2665)
      G_c mean: -0.1250 (σ=0.3307)
      JS div: 0.0032
      Δmeans: 0.0481 CI[-0.1266, 0.1771] 
    Google:  INSUFFICIENT_DATA

  ── C